# CO2Dot calibration run 3: a campaign of sessions (multi-device, LI-6800)

Protocol 3 generates the full calibration dataset as a **campaign** of sessions of at most 7 h. A campaign
folder holds `campaign.json` (grid, number of repeats, spectrometer settings frozen at the first session, and a
ledger of what has been measured) and one run folder per session. Every session: anchor + zero, the cells the
planner picks (least-measured first, at most two RH levels, dry before humid, temperatures in a random order,
cut to the time budget), anchor + zero. Each (RH, T) cell is measured `repeats` times, on different days and in
different orders, which is what separates film drift from the conditions. All logic is in `co2dot_calib3.py`
(on top of `co2dot_calib.py` for devices, LI-6800, loader, and `co2dot_calib2.py` for the auto-tune).

**Dose discipline.** LED 4 mA; logging at 15 s while approaching and dwelling, 5 s only inside the 60 s
window of every step. About a quarter of the daily dose that protocol 1 inflicted on the gen1 film.

**Per step.** Setpoint, wait within tolerance (zero below 15 ppm, top below 40 ppm, others 3 ppm or 1 %),
then dwell until the last 60 s of log10(630/515) are flat (minimum 1 min, 3 min at 65 % RH and above,
maximum 5 min), then the 60 s window. CO₂ levels 0, 200, 400, 800, 2000, 0 (+-30 ppm jitter on the middle
ones). Grid RH 20, 35, 50, 65, 80 % × T 17, 22, 27, 32 °C, minus the cells above the humidifier dew-point
ceiling of 19.5 °C or within 3 °C of condensation.

**Before every session**
1. Devices in the chamber facing the air stream; leaf thermocouple on the film; a blank device if available.
2. Cartridge, soda lime, desiccant and humidifier columns checked; PC sleep disabled.
3. First session only: `freeze_settings` auto-tunes gain/atime at 4 mA to ~30 000 counts. Later sessions
   re-apply the frozen settings; never re-tune inside a campaign, the counts must stay comparable.
4. Preflight must show a real LED signal, no channel above 60 % of 65 535, and noise below 1 %.

**How a session ends.** On its time budget or when its plan is complete; on the file `STOP` placed in the
session folder or the campaign folder; on the CO₂ watchdog (setpoints above ambient not reached twice in a row:
cartridge empty); or on the humidity watchdog (three RH timeouts in a row: desiccant or humidifier exhausted).
In all these cases the session measures the **end anchor pair** (anchor T/RH, 400 then 0 ppm) before restoring
the chamber, so the drift over the session can always be read from its start and end anchors; `campaign.json`
records `end_reason` and `end_anchor` per session, the stream and `cycles.jsonl` tag anchor steps with
`anchor: start|end`. Ctrl-C (interrupt the kernel) stops at once *without* the end anchor unless
`end_anchor_on_interrupt=True`; chamber restored, ledger and session plan kept; `run.run()` again resumes the
same session. `camp.status()` shows the ledger, how the last session ended and the next session's plan.

In [ ]:
import sys, time
from datetime import datetime
from pathlib import Path
# folder holding co2dot*.py, li_mqtt.py: this notebook's folder (Jupyter cwd) or <repo>/Scripts/Li-Control
LI_CONTROL_DIR = next((str(c) for c in (Path.cwd(), Path.cwd() / "Scripts" / "Li-Control")
                       if (c / "co2dot_calib3.py").exists()), None)
if LI_CONTROL_DIR is None:
    raise FileNotFoundError("co2dot_calib3.py not found near " + str(Path.cwd()))
if LI_CONTROL_DIR not in sys.path:
    sys.path.insert(0, LI_CONTROL_DIR)
from co2dot_calib import connect_dots, preflight, LicorLive, MockLicor, mock_dots, load_run, settled_points
from co2dot_calib3 import Protocol3, Campaign, Session

## 1. Campaign

In [ ]:
CAMPAIGN_DIR = Path(r"co2dot_data") / "campaign_TB-PET-PTFE_gen1"      # one folder per film / formulation
REPEATS = 3
P = Protocol3(
    rh_levels=(20, 35, 50, 65, 80), t_levels=(17, 22, 27, 32),
    co2_levels=(0, 200, 400, 800, 2000, 0), co2_jitter_ppm=30,
    max_dew_c=19.5, dewpoint_margin_c=3.0, cooldown_dry=True,
    session_budget_h=7.0, max_rh_levels_per_session=2,
    end_anchor_on_stop=True, end_anchor_on_interrupt=False,      # end anchor pair after STOP file, watchdogs, budget
    co2_watchdog_strikes=2, rh_watchdog_strikes=3,
    flow_umol_s=400,
)
camp = Campaign(CAMPAIGN_DIR, P, repeats=REPEATS)        # creates campaign.json, or opens it (then P is updated in it)
print(P.feasibility_table()); print(); print(camp.status())

## 2. Devices

In [ ]:
LABELS = {                      # device_id (USB serial = MAC) -> bench label; "blank" marks a device without dye
    "3C:DC:75:0D:FA:64": "A",
}
LED_MA = 4
dots = connect_dots(labels=LABELS)
if not camp.data["settings"]:
    settings = camp.freeze_settings(dots, led_ma=LED_MA, autotune_target=30000)    # first session of the campaign
else:
    settings = camp.apply_settings(dots)                                           # later sessions
print(settings)
report = preflight(dots, led_ma=LED_MA, n=3)

## 3. LI-6800

In [ ]:
from li_mqtt import LI6800
li = LI6800.connect(settle_s=15)
print("broker", li.broker, "topics", list(li._latest))
time.sleep(2)
print({k: li.get(k) for k in ("CO2_s", "CO2_r", "H2O_s", "Tchamber", "Txchg", "Tleaf", "RHcham", "Flow", "Press")})
licor = LicorLive(li)

## 4. Dry run without hardware (optional, ~3 min)

Two compressed sessions of a mock campaign in a temp folder: validates the planner, the ledger, the dwell
rule, the file format and the loader.

In [ ]:
# import tempfile, shutil
# tmp = Path(tempfile.gettempdir()) / "co2dot_campaign_mock"; shutil.rmtree(tmp, ignore_errors=True)
# lic = MockLicor(tau_s={"co2": 1, "t": 1, "rh": 1}); mdots = mock_dots(2, lic)
# mcamp = Campaign(tmp, Protocol3.quick(), repeats=2)
# mcamp.freeze_settings(mdots, led_ma=4, autotune_target=None, gain=6, atime=100, astep=999)
# for k in range(2): print(Session(mcamp, mdots, lic, seed=k).run())
# print(mcamp.status())

## 4b. Live streaming to openJII (optional)

Every logged cycle is also published to the openJII experiment over MQTT (AWS IoT Core, mTLS, QoS 1) on
`experiment/data_ingest/v1/<experiment>/generic/<sensorVersion>/<sensorId>`. The CO2Dot is registered as a
`generic` device under your account, its certificate is issued once and kept in `~/.openjii/devices/<serial>/`,
and the device is onboarded to the experiment; all of this is done through the REST API with your personal
API key, read from `~/.openjii/api_key` (or `OPENJII_API_KEY`). The key and the certificates never go into the
notebook, the repo or the data folder. Publishing is non-blocking: `cycles.jsonl` stays the complete record even
if the network drops. Set `STREAM = False` to run without it.

In [ ]:
STREAM = True
EXPERIMENT_ID = "81cb1484-372c-417e-ab12-57b016f40f02"      # openJII experiment "CO2Dot-calibration-data"
publisher = None
if STREAM:
    from openjii_stream import OpenJIIClient, OpenJIIPublisher
    client = OpenJIIClient.from_store()
    print("experiment:", client.experiment(EXPERIMENT_ID)["name"])
    serial = next(iter(dots))                                  # the first (non-blank) device streams; one message per cycle holds all devices
    cfg = client.provision(serial, EXPERIMENT_ID, name=f"CO2Dot {LABELS.get(serial, serial[-5:])}")
    publisher = OpenJIIPublisher(cfg, experiment_id=EXPERIMENT_ID, sensor_version="1.0",
                                 device_name=f"CO2Dot {LABELS.get(serial, '')}".strip(),
                                 extra={"output": "calibration campaign " + CAMPAIGN_DIR.name})
    publisher.connect()

## 5. Run one session (blocks this cell, at most `session_budget_h`)

In [ ]:
run = Session(camp, dots, licor, publisher=publisher,
              meta_extra={"operator": "", "notes": "devices in chamber facing the air stream; thermocouple on film A"})
print("session folder:", run.outdir); print("planned cells:", run.plan["cells"], f"~{run.plan['estimate_s']/3600:.1f} h")
status = run.run()
print("status:", status); print(camp.status())

In [ ]:
# Resume the same session after Ctrl-C or a cartridge swap (same run object -> same folder, plan and ledger):
# status = run.run(); print(status)
# After a kernel restart: re-run sections 1-3, then
# run = Session(camp, dots, licor, outdir=CAMPAIGN_DIR / "<session folder>"); status = run.run()

## 5b. Ending a session early, with the end anchor

Section 5 blocks this kernel, so an early stop is requested from outside it: create an empty file named `STOP`
in the session folder (`run.outdir`) or in the campaign folder. The session finishes the current step, measures
the end anchor pair, restores the chamber and returns `'stopped'`; the file is consumed. The CO₂ and humidity
watchdogs and the time budget end a session the same way. Ctrl-C returns `'interrupted'` without the end anchor
(unless `end_anchor_on_interrupt=True`); a second Ctrl-C during an end anchor aborts it (`end_anchor_aborted`).

In [ ]:
# From a terminal (PowerShell) while section 5 is running:
#     New-Item -ItemType File "<CAMPAIGN_DIR>\STOP"
# or from a second notebook / Python process:
#     from pathlib import Path; (Path(r"<CAMPAIGN_DIR>") / "STOP").touch()
# Afterwards, how the session ended and whether the end anchor was measured:
# import pandas as pd
# ev = pd.read_json(run.outdir / "events.jsonl", lines=True)
# ev[ev["kind"].isin(["budget_stop", "stop_requested", "cartridge_suspected", "humidity_suspected", "session_stopped",
#                     "session_done", "session_interrupted", "anchor_pair_done", "end_anchor_aborted"])]
# print(camp.status())        # last line: how the last session ended, end anchor yes/no

## 6. Quick look at this session

In [ ]:
import matplotlib.pyplot as plt
cyc, ev, meta = load_run(run.outdir)
print(cyc.shape, "samples x devices;", len(ev), "events")
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for lab, g in cyc.groupby("label"):
    ax[0].plot(g["ts"], g["lr_630_515"], ".", ms=3, label=lab)
    ax[2].plot(g["ts"], g["env_T"], ".", ms=2, label=f"{lab} board T"); ax[2].plot(g["ts"], g["env_RH"], ".", ms=2, label=f"{lab} board RH")
g0 = cyc.drop_duplicates("cycle")
ax[1].plot(g0["ts"], g0["licor_CO2_s"], "k.", ms=2); ax[1].set_ylabel("CO2_s (ppm)")
ax[2].plot(g0["ts"], g0["licor_Tchamber"], "k.", ms=2, label="Tchamber"); ax[2].plot(g0["ts"], g0["licor_RHcham"], "k.", ms=2, label="RHcham")
ax[0].set_ylabel("log10(630/515)"); ax[0].legend(ncol=4, fontsize=8); ax[2].legend(ncol=4, fontsize=8)
plt.tight_layout()
lit_cols = [c for c in cyc.columns if c.startswith("lit_")]
print("brightest lit channel, max over the session:", int(cyc[lit_cols].max().max()), "of 65535")
sp = settled_points(cyc, last_s=60)        # the 60 s windows (phase_transition == False)
sp[["phase_step", "label", "kind", "pass_", "t_set", "rh_set", "co2_set", "n", "licor_CO2_s", "env_T", "env_RH", "lr_630_515"]].round(3)

## 7. Close

In [ ]:
run.close()        # closes every CO2Dot port
li.close()
if publisher is not None:
    publisher.close()      # flushes the queue, prints sent / failed / dropped counts